## Import Libraries

In [1]:
import arcpy 
import pandas as pd 
import numpy as np
import os
import glob as gb

## Set Directory

In [2]:
path = 'C:\\Users\\Thepr\\Documents\\Consulting'
os.chdir(path)

In [3]:
workingfolder = r'C:\Users\Thepr\Documents\Consulting\CityofRedWing'
projectGDB = os.path.join(workingfolder, 'ArcGISPro', 'RedWing.gdb')
outputGDB = os.path.join(workingfolder, 'Soft Parcel Analysis RedWing', 'SoftParcelAnalysis.gdb')


arcpy.env.workspace = projectGDB
arcpy.env.overwriteOutput = True

spatialreference = arcpy.SpatialReference(26859)

arcpy.env.outputCoordinateSystem = spatialreference

print(f'The arcpy version being use is {arcpy.GetInstallInfo()["Version"]}')

The arcpy version being use is 3.2


## Join Land value data to Parcel using Pin

In [4]:
parcel = os.path.join(outputGDB,'Parcels')
appraisal = os.path.join(outputGDB,'AppraisalCityofRedWing')

In [5]:
excludefields = ['OBJECTID', 'PIN']

appraisalfields = []

for field in arcpy.ListFields(appraisal):
    if field.name not in excludefields:
        appraisalfields.append(field.name)

In [6]:
parcelfieldnames = [row.name for row in arcpy.ListFields(parcel)]

for field in arcpy.ListFields(appraisal):
    
    if field.name in excludefields:
        continue

    if field.name not in parcelfieldnames:

        if field.type == "String":
            arcpy.management.AddField(parcel,field.name,"TEXT",field_length=field.length)

        elif field.type in ["Integer", "SmallInteger"]:
            arcpy.management.AddField(parcel, field.name, "LONG")

        elif field.type == "Double":
            arcpy.management.AddField(parcel, field.name, "DOUBLE")

        elif field.type == "Single":
            arcpy.management.AddField(parcel, field.name, "FLOAT")

        elif field.type == "Date":
            arcpy.management.AddField(parcel, field.name, "DATE")

In [7]:
appraisaldict = {}
with arcpy.da.SearchCursor(appraisal, ['PIN'] + appraisalfields) as cursor:
    for row in cursor:
        pin = row[0]
        if pin not in appraisaldict:
            appraisaldict[pin] = tuple(row[1:])

matched = 0
unmatched = 0
with arcpy.da.UpdateCursor(parcel, ['PIN'] + appraisalfields) as cursor:
    for row in cursor:
        pin = row[0]
        if pin in appraisaldict:
            row[1:] = appraisaldict[pin]
            matched += 1
        else:
            row[1:] = [None] * len(appraisalfields)
            unmatched += 1
        cursor.updateRow(row)

print(f"Matched PINs: {matched}")
print(f"Unmatched PINs: {unmatched}")

Matched PINs: 8261
Unmatched PINs: 114


### Area Weighted Zoning Score

In [8]:
parcel = os.path.join(outputGDB,'Parcels')
zoning = os.path.join(projectGDB,'ZoningRW')
sitezoning = os.path.join(outputGDB,'ZoningDistrictsParcel')

In [9]:
if arcpy.Exists(sitezoning):
    arcpy.Delete_management(sitezoning)

arcpy.analysis.Intersect(
    in_features=[parcel, zoning],
    out_feature_class=sitezoning,
    join_attributes="ALL")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\Soft Parcel Analysis RedWing\\SoftParcelAnalysis.gdb\\ZoningDistrictsParcel'>

In [10]:
arcpy.management.CalculateGeometryAttributes(sitezoning, "FragmentAcres AREA", "", "ACRES",arcpy.SpatialReference(102738))

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\Soft Parcel Analysis RedWing\\SoftParcelAnalysis.gdb\\ZoningDistrictsParcel'>

In [11]:
zonescores = {
    'B3-Central Business':                               1.0,
    'B2-General Business':                               0.9,
    'B1-Local Business':                                 0.8,
    'MC-Mixed Use/Industrial/Office Commercial':         1.0,
    'MCT-Mixed Use Commercial Tourism':                  1.0,
    'RF-Riverfront':                                     0.8,
    'I1-Light Industrial':                               0.8,
    'I2-General Industrial':                             0.7,
    'CI-Civic':                                          0.5,
    'RM2-Residential Multi-Family Two (+16 units/acre)': 1.0,
    'RM1-Residential Multi-Family One (8-16 units/acre)':1.0,
    'R2-Residential Two (5-8 units/acre)':               1.0,
    'R1-Residential One (3.5-5 units/acre)':             0.3,
    'R1-Residential one (3.5-5 units/acre)':             0.3,
    'AR-Agriculture Residential':                        0.2,
    'AC-Agriculture Conservation':                       0.1,
    'A-Agriculture':                                     0.1,
    'Prairie Island Indian Community':                   0.0,
}

In [12]:
arcpy.management.AddField(sitezoning, "ZoneDocScore", "DOUBLE")

with arcpy.da.UpdateCursor(sitezoning, ["ZONE_DOC", "ZoneDocScore"]) as cursor:
    for row in cursor:
        row[1] = zonescores.get(row[0], 0.3)
        cursor.updateRow(row)

In [13]:
updated = 0
altupdated = 0

weightedtotals = {}

with arcpy.da.SearchCursor(sitezoning, ["PIN", "FragmentAcres", "ZoneDocScore"]) as cursor:
    for row in cursor:
        pin, acres, score = row[0], row[1], row[2]
        if pin not in weightedtotals:
            weightedtotals[pin] = [0.0, 0.0]
        weightedtotals[pin][0] += acres * score
        weightedtotals[pin][1] += acres


arcpy.management.AddField(parcel, "ZoneDocScore_Weighted", "DOUBLE")

with arcpy.da.UpdateCursor(parcel, ["PIN", "ZoneDocScore_Weighted"]) as cursor:
    for row in cursor:
        pin = row[0]
        if pin in weightedtotals and weightedtotals[pin][1] > 0:
            row[1] = round(weightedtotals[pin][0] / weightedtotals[pin][1], 4)
            updated += 1
        else:
            row[1] = 0.3
            altupdated += 1
        cursor.updateRow(row)
        
print(f"Updated PINs: {updated}")
print(f"Not Updated PINs: {altupdated}")

Updated PINs: 7743
Not Updated PINs: 632


## Land Use Weight Score

In [14]:
landuse = os.path.join(projectGDB,'LU_DevStatusRW')
sitelanduse = os.path.join(outputGDB, 'SiteLUIntersect')
sitelandusestats = os.path.join(projectGDB, 'SiteLUStats')

In [15]:
landusescores = {
    'Vacant':                    1.0,
    'Downtown':                  1.0,
    'Mixed Use Commercial':      1.0,
    'Community Commercial':      0.9,
    'Regional Commercial':       0.8,
    'Industry':                  0.7,
    'High Density Residential':  0.7,
    'Medium Density Residential':0.5,
    'Institutional':             0.4,
    'Utility':                   0.4,
    'Park (active)':             0.2,
    'Open Space':                0.2,
    'Low Density Residential':   0.2,
    'Rural Residential':         0.1,
    'Agriculture':               0.1,
    'Prairie Island Community':  0.0,
}

In [16]:
arcpy.analysis.Intersect(
    in_features=[parcel, landuse],
    out_feature_class=sitelanduse,
    join_attributes="ALL")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\Soft Parcel Analysis RedWing\\SoftParcelAnalysis.gdb\\SiteLUIntersect'>

In [17]:
arcpy.management.CalculateGeometryAttributes(sitelanduse, "FragmentAcres AREA", "", "ACRES",arcpy.SpatialReference(102738))

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\Soft Parcel Analysis RedWing\\SoftParcelAnalysis.gdb\\SiteLUIntersect'>

In [18]:
arcpy.management.AddField(sitelanduse, "LandUseScore", "DOUBLE")

with arcpy.da.UpdateCursor(sitelanduse, ["LANDUSEDES", "LandUseScore"]) as cursor:
    for row in cursor:
        row[1] = landusescores.get(row[0], 0.3)
        cursor.updateRow(row)

In [19]:
updated = 0
altupdated = 0
weightedtotals = {}

with arcpy.da.SearchCursor(sitelanduse, ["PIN", "FragmentAcres", "LandUseScore"]) as cursor:
    for row in cursor:
        pin, acres, score = row[0], row[1], row[2]
        if pin not in weightedtotals:
            weightedtotals[pin] = [0.0, 0.0]
        weightedtotals[pin][0] += acres * score
        weightedtotals[pin][1] += acres

arcpy.management.AddField(parcel, "LandUseScoreWeighted", "DOUBLE")

with arcpy.da.UpdateCursor(parcel, ["PIN", "LandUseScoreWeighted"]) as cursor:
    for row in cursor:
        pin = row[0]
        if pin in weightedtotals and weightedtotals[pin][1] > 0:
            row[1] = round(weightedtotals[pin][0] / weightedtotals[pin][1], 4)
            updated += 1
        else:
            row[1] = 0.3
            altupdated += 1
        cursor.updateRow(row)

print(f"Updated TAXPINs: {updated}")
print(f"Not Updated TAXPINs: {altupdated}")

Updated TAXPINs: 7754
Not Updated TAXPINs: 621


### ILR

In [20]:
arcpy.management.AddField(parcel, "ILR", "DOUBLE")

with arcpy.da.UpdateCursor(parcel, ["TPBLDG", "TPLAND", "ILR"]) as cursor:
    for row in cursor:
        imp  = row[0]
        land = row[1]
        if land is not None and land > 0:
            row[2] = round(imp / land, 2) if imp is not None else 0.0
        else:
            row[2] = None
        cursor.updateRow(row)

print("ILR calculated.")

ILR calculated.


### Soft Parcels

In [21]:
arcpy.management.AddField(parcel, "SoftParcel", "TEXT", field_length=20)

with arcpy.da.UpdateCursor(parcel, ["ILR", "ZoneDocScore_Weighted", "LandUseScoreWeighted", 
                                    "YR_BUILT", "SoftParcel"]) as cursor:
    for row in cursor:
        ilr       = row[0]
        zonescore = row[1]
        luscore   = row[2]
        yr_built  = row[3]

        if (ilr is not None and ilr <= 0.5 and
            zonescore is not None and zonescore >= 0.5 and
            luscore is not None and luscore >= 0.5 and
            (yr_built is None or yr_built == 0 or yr_built <= 2005)):
            row[4] = "Soft Parcel"
        else:
            row[4] = "Not Soft Parcel"
        cursor.updateRow(row)

softcount = 0
with arcpy.da.SearchCursor(parcel, ["SoftParcel"]) as cursor:
    for row in cursor:
        if row[0] == "Soft Parcel":
            softcount += 1

print(f"Soft Parcels: {softcount}")

Soft Parcels: 459


In [22]:
ilr_count   = 0
zone_count  = 0
lu_count    = 0
all_count   = 0

with arcpy.da.SearchCursor(parcel, ["ILR", "ZoneDocScore_Weighted", "LandUseScoreWeighted"]) as cursor:
    for row in cursor:
        ilr, zone, lu = row[0], row[1], row[2]
        if ilr is not None and ilr <= 0.5:
            ilr_count += 1
        if zone is not None and zone >= 0.5:
            zone_count += 1
        if lu is not None and lu >= 0.5:
            lu_count += 1
        if (ilr is not None and ilr <= 0.5 and
            zone is not None and zone >= 0.5 and
            lu is not None and lu >= 0.5):
            all_count += 1

print(f"ILR passes: {ilr_count}")
print(f"Zone passes: {zone_count}")
print(f"LU passes: {lu_count}")
print(f"All pass: {all_count}")

ILR passes: 1606
Zone passes: 4396
LU passes: 4426
All pass: 461


In [23]:
with arcpy.da.SearchCursor(parcel, ["ILR", "ZoneDocScore_Weighted", "TPCLS1_DSC"]) as cursor:
    for row in cursor:
        ilr, zone, cls = row[0], row[1], row[2]
        if ilr is not None and ilr <= 0.5:
            print(f"ILR: {ilr} | ZoneScore: {zone} | Class: {cls}")

ILR: 0.0 | ZoneScore: 0.3 | Class: 4B4 UNIMPROVED RESIDENTIAL LAND
ILR: 0.0 | ZoneScore: 0.3 | Class: 1A/1B/4BB RESIDENTIAL SINGLE UNIT
ILR: 0.0 | ZoneScore: 0.3 | Class: 1A/1B/4BB RESIDENTIAL SINGLE UNIT
ILR: 0.0 | ZoneScore: 0.3 | Class: 4B4 UNIMPROVED RESIDENTIAL LAND
ILR: 0.0 | ZoneScore: 0.3 | Class: 5E COUNTY-PUBLIC SERVICE-OTHER
ILR: 0.0 | ZoneScore: 0.3 | Class: 4B4 UNIMPROVED RESIDENTIAL LAND
ILR: 0.0 | ZoneScore: 0.3 | Class: 5E STATE ACQUIRED - PILT
ILR: 0.0 | ZoneScore: 0.3 | Class: 2B/1B RURAL VACANT LAND
ILR: 0.0 | ZoneScore: 0.3 | Class: 2B/1B RURAL VACANT LAND
ILR: 0.0 | ZoneScore: 0.3 | Class: 5E STATE ACQUIRED - PILT
ILR: 0.0 | ZoneScore: 0.3 | Class: 2B/1B RURAL VACANT LAND
ILR: 0.0 | ZoneScore: 0.3 | Class: 2B/1B RURAL VACANT LAND
ILR: 0.0 | ZoneScore: 0.3 | Class: 5E STATE ACQUIRED - PILT
ILR: 0.0 | ZoneScore: 0.3 | Class: 5E STATE ACQUIRED - PILT
ILR: 0.06 | ZoneScore: 0.3 | Class: 2A/1B/4BB AGRICULTURAL
ILR: 0.0 | ZoneScore: 0.3 | Class: 2B/1B RURAL VACANT LAND
I

### Clusters 

In [24]:
softparcels = os.path.join(outputGDB, 'SoftParcels')

if arcpy.Exists(softparcels):
    arcpy.Delete_management(softparcels)

arcpy.conversion.ExportFeatures(
    parcel,
    softparcels,
    "SoftParcel = 'Soft Parcel'",
    "NOT_USE_ALIAS",
    None,
    None)

recordscount = int(arcpy.management.GetCount(softparcels)[0])
print(f"Soft Parcels exported: {recordscount}")

Soft Parcels exported: 459


### Aggregate Polygon 

In [25]:
softparcelsclusters = os.path.join(outputGDB, 'SoftParcelsClusters')
softparcelsclustertable = os.path.join(outputGDB, 'SoftParcelsClustersTable')

if arcpy.Exists(softparcelsclusters):
    arcpy.Delete_management(softparcelsclusters)

if arcpy.Exists(softparcelsclustertable):
    arcpy.Delete_management(softparcelsclustertable)

arcpy.cartography.AggregatePolygons(
    softparcels,
    softparcelsclusters,
    "500 Feet",
    "0 SquareFeetUS",
    "0 SquareFeetUS",
    "NON_ORTHOGONAL",
    None,
    softparcelsclustertable,
    None)

recordscount = int(arcpy.management.GetCount(softparcelsclusters)[0])
print(f"Clusters: {recordscount}")

Clusters: 42


### Calculate Cluster Acreage

In [26]:
arcpy.management.AddField(softparcelsclusters, "ClusterAcreage", "DOUBLE")

arcpy.management.CalculateGeometryAttributes(
    softparcelsclusters,
    "ClusterAcreage AREA",
    "",
    "ACRES",
    arcpy.SpatialReference(102738),
    "SAME_AS_INPUT")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\Soft Parcel Analysis RedWing\\SoftParcelAnalysis.gdb\\SoftParcelsClusters'>

###  Minimum Bounding Geometry

In [27]:
softparcelscircles = os.path.join(outputGDB, 'SoftParcelsClustersCircles')

if arcpy.Exists(softparcelscircles):
    arcpy.Delete_management(softparcelscircles)

arcpy.management.MinimumBoundingGeometry(
    softparcelsclusters,
    softparcelscircles,
    "CIRCLE",
    "NONE",
    None,
    "NO_MBG_FIELDS")

recordscount = int(arcpy.management.GetCount(softparcelscircles)[0])
print(f"Cluster circles: {recordscount}")

Cluster circles: 42


### DBSCAN

In [28]:
softparcelspoints = os.path.join(outputGDB, 'SoftParcelsPoints')

if arcpy.Exists(softparcelspoints):
    arcpy.Delete_management(softparcelspoints)

arcpy.management.FeatureToPoint(softparcels, softparcelspoints, "CENTROID")

softparcelsdensity = os.path.join(outputGDB, 'SoftParcelsDensityCluster')

if arcpy.Exists(softparcelsdensity):
    arcpy.Delete_management(softparcelsdensity)

arcpy.stats.DensityBasedClustering(
    softparcelspoints,
    softparcelsdensity,
    "DBSCAN",
    2,
    "500 Feet",
    None,
    None,
    None)

recordscount = int(arcpy.management.GetCount(softparcelsdensity)[0])
print(f"DBSCAN output: {recordscount}")

DBSCAN output: 459


### Filter Clustered Points and Join Back to Polygons

In [29]:
clustered = arcpy.management.SelectLayerByAttribute(
    softparcelsdensity,
    "NEW_SELECTION",
    "COLOR_ID > 0",
    None)

recordscount = int(arcpy.management.GetCount(clustered)[0])
print(f"Clustered points: {recordscount}")

softparcelspreferred = os.path.join(outputGDB, 'SoftParcelsPreferred')


Clustered points: 417


In [30]:
if arcpy.Exists(softparcelspreferred):
    arcpy.Delete_management(softparcelspreferred)
    
arcpy.analysis.SpatialJoin(
    softparcels,
    clustered,
    softparcelspreferred,
    "JOIN_ONE_TO_ONE",
    "KEEP_COMMON",
    '',
    "COMPLETELY_CONTAINS",
    None,
    "",
    None)

recordscount = int(arcpy.management.GetCount(softparcelspreferred)[0])
print(f"Preferred soft parcels: {recordscount}")

Preferred soft parcels: 412


### Minimum Bounding Geometry by Cluster

In [31]:
softparcelsboundary = os.path.join(outputGDB, 'SoftParcelsBoundary')

if arcpy.Exists(softparcelsboundary):
    arcpy.Delete_management(softparcelsboundary)

arcpy.management.MinimumBoundingGeometry(
    softparcelspreferred,
    softparcelsboundary,
    "CIRCLE",
    "LIST",
    "CLUSTER_ID;COLOR_ID",
    "NO_MBG_FIELDS")

recordscount = int(arcpy.management.GetCount(softparcelsboundary)[0])
print(f"Cluster boundaries: {recordscount}")

Cluster boundaries: 50


### Export Parcels

###  Convert parcels to centroids

In [32]:
parcelcentroids = os.path.join(outputGDB, 'ParcelCentroids')

if arcpy.Exists(parcelcentroids):
    arcpy.Delete_management(parcelcentroids)

arcpy.management.FeatureToPoint(parcel, parcelcentroids, "CENTROID")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\Soft Parcel Analysis RedWing\\SoftParcelAnalysis.gdb\\ParcelCentroids'>

###  Spatial join centroids to zoning

In [33]:
centroidzoning = os.path.join(outputGDB, 'ParcelCentroidsZoning')

if arcpy.Exists(centroidzoning):
    arcpy.Delete_management(centroidzoning)

arcpy.analysis.SpatialJoin(parcelcentroids,zoning,centroidzoning,"JOIN_ONE_TO_ONE","KEEP_ALL",'', "INTERSECT",None,"",None)

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\Soft Parcel Analysis RedWing\\SoftParcelAnalysis.gdb\\ParcelCentroidsZoning'>

### Build zoning lookup and write back to parcel

In [34]:
zoningresults = {}
with arcpy.da.SearchCursor(centroidzoning, ["PIN", "ZONE_DOC"]) as cursor:
    for row in cursor:
        zoningresults[row[0]] = row[1]
        
arcpy.management.AddField(parcel, "ZONE_DOC", "TEXT", field_length=100)
arcpy.management.AddField(parcel, "ZoneDocScore", "DOUBLE")

with arcpy.da.UpdateCursor(parcel, ["PIN", "ZONE_DOC", "ZoneDocScore"]) as cursor:
    for row in cursor:
        pin  = row[0]
        zone = zoningresults.get(pin, None)
        row[1] = zone
        row[2] = zonescores.get(zone, 0.3) if zone is not None else 0.3
        cursor.updateRow(row)

print("Zoning assigned.")

Zoning assigned.


###  Spatial join centroids to land use

In [35]:
centroidlanduse = os.path.join(outputGDB, 'ParcelCentroidsLandUse')

if arcpy.Exists(centroidlanduse):
    arcpy.Delete_management(centroidlanduse)

arcpy.analysis.SpatialJoin(parcelcentroids,landuse,centroidlanduse,"JOIN_ONE_TO_ONE","KEEP_ALL",'',"INTERSECT",None,"",None)

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\Soft Parcel Analysis RedWing\\SoftParcelAnalysis.gdb\\ParcelCentroidsLandUse'>

### Build land use lookup and write back to parcel

In [36]:
landusesresults = {}
with arcpy.da.SearchCursor(centroidlanduse, ["PIN", "LANDUSEDES"]) as cursor:
    for row in cursor:
        landusesresults[row[0]] = row[1]

arcpy.management.AddField(parcel, "LANDUSEDES", "TEXT", field_length=100)
arcpy.management.AddField(parcel, "LandUseScore", "DOUBLE")

with arcpy.da.UpdateCursor(parcel, ["PIN", "LANDUSEDES", "LandUseScore"]) as cursor:
    for row in cursor:
        pin = row[0]
        lu  = landusesresults.get(pin, None)
        row[1] = lu
        row[2] = landusescores.get(lu, 0.3) if lu is not None else 0.3
        cursor.updateRow(row)

print("Land use assigned.")

Land use assigned.


###  Export soft parcels with only analysis fields

In [37]:
softparcels = os.path.join(outputGDB, 'SoftParcels')

if arcpy.Exists(softparcels):
    arcpy.Delete_management(softparcels)

fieldmapping = arcpy.FieldMappings()
fieldmapping.addTable(parcel)

analysisfields = [
    'PIN', 'PIN_TEXT',
    'PHYSADDR', 'PHYSCITY', 'PHYSZIP',
    'OWNNAME', 'OWADR1',
    'MAINACRES',
    'TPLAND', 'TPBLDG', 'ESTTOTVAL',
    'YR_BUILT', 'EFF_YR_BLT',
    'TPCLS1_DSC', 'TPCLS2_DSC',
    'HMST_DSC1', 'DELINQUENT', 'TX_EX_FLAG',
    'SALE_DATE', 'SALE_PRICE',
    'ZONE_DOC', 'ZoneDocScore',
    'LANDUSEDES', 'LandUseScore',
    'ILR', 'SoftParcel'
]

for field in fieldmapping.fields:
    if field.name not in analysisfields:
        fieldmapping.removeFieldMap(fieldmapping.findFieldMapIndex(field.name))

arcpy.conversion.ExportFeatures(
    parcel,
    softparcels,
    "SoftParcel = 'Soft Parcel'",
    "NOT_USE_ALIAS",
    fieldmapping,
    None)

recordscount = int(arcpy.management.GetCount(softparcels)[0])
print(f"Soft parcels exported: {recordscount}")

Soft parcels exported: 459
